# Statistical Forecasting Models

Companion code for the *Predictive Analytics Lab* (section 26). Run the cell below.

In [ ]:
# SMA, WMA, EMA, and an ARIMA-style model on the airline series (train 1949-1958, test 1959-1960)
import numpy as np
import matplotlib.pyplot as plt
AIR = [112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
y = np.array(AIR, dtype=float); tr, te = y[:120], y[120:]
rmse = lambda f: np.sqrt(np.mean((f - te) ** 2))

k = 12
sma = tr[-k:].mean()
w = np.arange(1, k + 1); wma = np.sum(w * tr[-k:]) / w.sum()
alpha = 0.3; ema = tr[0]
for v in tr[1:]: ema = alpha * v + (1 - alpha) * ema
for name, v in [("SMA", sma), ("WMA", wma), ("EMA", ema)]:
    print(f"{name}: flat forecast {v:.1f}, test RMSE {rmse(np.full(24, v)):.2f}")

# ARIMA-style: log transform, seasonal difference (lag 12), AR(1) on what is left
z = np.log(tr); s = z[12:] - z[:-12]
mu = s.mean(); phi = np.sum((s[1:] - mu) * (s[:-1] - mu)) / np.sum((s[:-1] - mu) ** 2)
hist = list(z); sd = list(s)
for h in range(24):
    nxt = mu + phi * (sd[-1] - mu); sd.append(nxt); hist.append(hist[-12] + nxt)
arima = np.exp(hist[120:])
print(f"ARIMA-style (AR(1) on seasonal log-differences): test RMSE {rmse(arima):.2f}")

plt.plot(y, c="gray", label="actual"); plt.plot(range(120, 144), arima, label="ARIMA-style")
plt.axhline(sma, xmin=120 / 144, ls="--", label="SMA flat forecast"); plt.legend(); plt.show()
# In Colab you can compare with statsmodels.tsa.arima.model.ARIMA and the prophet package.